### [GitHub](https://github.com/msultanmahmud/Stress_WESAD/blob/master/Multiclass_Pipeline_Individual_sub_INTEC.ipynb)

In [2]:
import numpy as np
import pandas as pd

In [3]:
# ! pip list

In [4]:
# ! pip install lightgbm

In [5]:
import numpy as np
import pandas as pd
import os
import sys
import random
import warnings
warnings.filterwarnings('ignore')
import pickle
from sklearn.model_selection import train_test_split
from sklearn import svm, metrics,preprocessing
#from sklearn import datasets
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,roc_curve, auc,classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_predict
from matplotlib import pyplot as plt
from collections import Counter
from scipy.stats import norm
import seaborn as sns; sns.set(font_scale=1.2)
%matplotlib inline
import time
import xgboost as xgb
import lightgbm as lgb

#### Redaing file

In [7]:
# with open("wsc_visit1_allsub.pkl", "rb") as f:
#     df = pickle.load(f)
#     df=df.sort_values(by='subject')

In [8]:
df=pd.read_csv('dreamt_preprocessed.csv')
print(df.shape)
df.subject.nunique()

(95147, 289)


100

In [9]:
# Identify columns to drop if they exist
columns_to_drop = [col for col in df.columns if 'skewness' in col.lower() or 'kurtosis' in col.lower()]

# Drop only those columns that actually exist
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
df 

,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
0,S002,1,3.0,-0.000008,0.000008,6.380306e-09,0.000002,-0.000004,-1.318379e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
1,S002,2,3.0,-0.000006,0.000005,4.531319e-08,0.000002,-0.000003,-1.123064e-06,-4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
2,S002,3,3.0,-0.000004,0.000005,4.547595e-08,0.000002,-0.000002,-9.277485e-07,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
3,S002,4,3.0,-0.000017,0.000019,6.071056e-08,0.000003,-0.000004,-1.220722e-06,1.464866e-07,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
4,S002,5,3.0,-0.000008,0.000009,6.067801e-08,0.000002,-0.000004,-1.416037e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-0.000025,0.000045,2.679077e-08,0.000008,-0.000012,-4.638743e-06,-1.464866e-07,...,72.12,0.859375,0.906250,0.884599,0.016580,0.859375,0.859375,0.890625,0.890625,0.906250
95143,S103,900,1.0,-0.000035,0.000032,1.027685e-07,0.000008,-0.000014,-4.638743e-06,1.464866e-07,...,68.87,0.812500,0.890625,0.869016,0.019533,0.828125,0.859375,0.859375,0.890625,0.890625
95144,S103,901,1.0,-0.000083,0.000080,1.070654e-07,0.000011,-0.000013,-4.541085e-06,3.906310e-07,...,69.03,0.843750,0.906250,0.871021,0.019181,0.843750,0.859375,0.875000,0.875000,0.906250
95145,S103,902,1.0,-0.000049,0.000056,8.274866e-08,0.000009,-0.000013,-4.931716e-06,1.464866e-07,...,69.42,0.734375,0.937500,0.869594,0.061489,0.734375,0.875000,0.890625,0.906250,0.921875


## Taking the cleaned final data

In [11]:
df = df[~df['subject'].isin(['S062', 'S097'])]
pre_procss_all=df.copy()
pre_procss_all

,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
0,S002,1,3.0,-0.000008,0.000008,6.380306e-09,0.000002,-0.000004,-1.318379e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
1,S002,2,3.0,-0.000006,0.000005,4.531319e-08,0.000002,-0.000003,-1.123064e-06,-4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
2,S002,3,3.0,-0.000004,0.000005,4.547595e-08,0.000002,-0.000002,-9.277485e-07,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
3,S002,4,3.0,-0.000017,0.000019,6.071056e-08,0.000003,-0.000004,-1.220722e-06,1.464866e-07,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
4,S002,5,3.0,-0.000008,0.000009,6.067801e-08,0.000002,-0.000004,-1.416037e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-0.000025,0.000045,2.679077e-08,0.000008,-0.000012,-4.638743e-06,-1.464866e-07,...,72.12,0.859375,0.906250,0.884599,0.016580,0.859375,0.859375,0.890625,0.890625,0.906250
95143,S103,900,1.0,-0.000035,0.000032,1.027685e-07,0.000008,-0.000014,-4.638743e-06,1.464866e-07,...,68.87,0.812500,0.890625,0.869016,0.019533,0.828125,0.859375,0.859375,0.890625,0.890625
95144,S103,901,1.0,-0.000083,0.000080,1.070654e-07,0.000011,-0.000013,-4.541085e-06,3.906310e-07,...,69.03,0.843750,0.906250,0.871021,0.019181,0.843750,0.859375,0.875000,0.875000,0.906250
95145,S103,902,1.0,-0.000049,0.000056,8.274866e-08,0.000009,-0.000013,-4.931716e-06,1.464866e-07,...,69.42,0.734375,0.937500,0.869594,0.061489,0.734375,0.875000,0.890625,0.906250,0.921875


In [12]:
# list(pre_procss_all.columns)

In [13]:
# ch=(237-3)/9
total_colum=8*9+3 ## 8 sensors *9 features
total_colum

75

## >>>>>> PSG DATA Separate <<<<<<<<<<<<

In [15]:
wanted_keywords = ['BVP', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR', 'IBI']
pattern = '|'.join(wanted_keywords)
 
# Select columns that do NOT match the regex pattern
df_psg = pre_procss_all.loc[:, ~pre_procss_all.columns.str.contains(pattern, case=False, regex=True)]
df_psg
# Drop columns with any null (NaN) values
df_psg_cleaned = df_psg.dropna(axis=1, how='any')
# Optionally: print removed columns
psg_removed_cols = df_psg.columns[df_psg.isnull().any()].tolist()
print("Removed columns with nulls:", psg_removed_cols)
df_psg_cleaned
 
df_psg_cleaned = df_psg_cleaned[df_psg_cleaned['label'].isin([0,1,2])]
df_psg_cleaned.head(3)

Removed columns with nulls: []


,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,RAT_p95,SAO2_min,SAO2_max,SAO2_mean,SAO2_std,SAO2_p5,SAO2_p25,SAO2_p50,SAO2_p75,SAO2_p95
305,S002,307,0.0,-0.000043,0.000020,-2.770550e-07,0.000007,-0.000010,-0.000003,4.394598e-07,...,0.000007,0.969907,0.969941,0.969924,0.000006,0.969915,0.969919,0.969924,0.969930,0.969934
306,S002,308,0.0,-0.000018,0.000014,6.477963e-09,0.000005,-0.000007,-0.000003,4.882887e-08,...,0.000005,0.958276,0.971334,0.966985,0.004638,0.959652,0.959699,0.969916,0.969931,0.969956
307,S002,309,0.0,-0.000050,0.000046,2.981816e-08,0.000008,-0.000011,-0.000003,2.441444e-07,...,0.000006,0.969916,0.969933,0.969924,0.000004,0.969918,0.969920,0.969924,0.969929,0.969931


In [ ]:
## >>>>>> PSG DATA Separation done <<<<<<<<<<<<

In [17]:
df=df_psg_cleaned.copy()

In [18]:
# df1=df_fusion_cleaned.head(1000)
# df1.head(3)
df1=df 

In [19]:
# df1=df_sample.copy()

In [20]:
# !jupyter kernelspec list

In [21]:
from joblib import Parallel, delayed
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
import pandas as pd
import numpy as np

def preprocess_scaling(df):
    '''Scales features and splits into train/test.'''
    X = df.drop(columns=['subject', 'epoch', 'label'])
    y = df['label']
    X_scaled = preprocessing.scale(X)
    return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name, auc_mode='ovr'):
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

        # AUC calculation — multi-class
        if hasattr(model, "predict_proba"):
            y_probs = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_probs, multi_class=auc_mode, average='macro')
        else:
            auc = None  # SVM without probability=True will not support this

        return (model_name, acc, f1, precision, recall, auc)
    except Exception as e:
        print(f"Error with model {model_name}: {e}")
        return (model_name, None, None, None, None, None)

def multiple_model(df, auc_mode='ovr'):
    X_train, X_test, y_train, y_test = preprocess_scaling(df)
    model_name_list = ['KNN', 'DT', 'XGBoost', 'LightGBM', 'RF', 'SVM']

    # SVM needs probability=True for predict_proba
    models = [
        KNeighborsClassifier(metric='minkowski', n_neighbors=2, weights='distance', p=2),
        DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=50),
        xgb.XGBClassifier(objective='multi:softprob', num_class=len(np.unique(y_train)), random_state=42,
                          max_depth=15, learning_rate=0.3, gamma=0.2, colsample_bytree=0.7,
                          min_child_weight=1, eval_metric='mlogloss', n_jobs=-1),
        lgb.LGBMClassifier(colsample_bytree=0.7, learning_rate=0.3, max_depth=100,
                           min_child_weight=0.5, n_estimators=200, verbose=-1, n_jobs=-1),
        RandomForestClassifier(max_depth=40, max_features='log2', n_estimators=200, n_jobs=-1),
        svm.SVC(C=10, kernel='rbf', degree=3, gamma=0.1, probability=True)  # Required for AUC
    ]

    results_list = Parallel(n_jobs=-1)(delayed(train_and_evaluate)(
        model, X_train, y_train, X_test, y_test, name, auc_mode
    ) for model, name in zip(models, model_name_list))

    results_df = pd.DataFrame(results_list, columns=['Model', 'Accuracy', 'F1', 'Precision', 'Recall', 'AUC'])
    results_df.set_index('Model', inplace=True)
    results_df=results_df[['Accuracy','AUC','Precision', 'Recall','F1']]
    return results_df

In [22]:
# # Default AUC = 'ovr'
# results = multiple_model(df1)
# results

In [23]:
# # Or try one-vs-one AUC if OvR seems low
# results_ovo = multiple_model(df1, auc_mode='ovo')
# results_ovo

### Ensemble Learning

In [25]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm

def ensemble_model(df, voting_type='hard', auc_mode='ovr'):
    X_train, X_test, y_train, y_test = preprocess_scaling(df)

    # Define base models
    knn = KNeighborsClassifier(metric='minkowski', n_neighbors=2, weights='distance', p=2)
    dt = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=50)
    xgb_model = xgb.XGBClassifier(objective='multi:softprob', random_state=42, min_child_weight=1,
                                  max_depth=15, learning_rate=0.3, gamma=0.2, colsample_bytree=0.7,
                                  eval_metric='mlogloss', n_jobs=-1, use_label_encoder=False)
    lgb_model = lgb.LGBMClassifier(colsample_bytree=0.7, learning_rate=0.3, max_depth=100,
                                   min_child_weight=0.5, n_estimators=200, verbose=-1, n_jobs=-1)
    rf = RandomForestClassifier(max_depth=40, max_features='log2', n_estimators=200, n_jobs=-1)
    svc = svm.SVC(C=10, kernel='rbf', degree=3, gamma=0.1, probability=(voting_type == 'soft'))

    # Create ensemble using voting
    ensemble = VotingClassifier(
        estimators=[
            ('KNN', knn),
            ('DT', dt),
            ('XGB', xgb_model),
            ('LGBM', lgb_model),
            ('RF', rf),
            ('SVM', svc)
        ],
        voting=voting_type,
        n_jobs=-1
    )

    ensemble.fit(X_train, y_train)
    y_pred = ensemble.predict(X_test)

    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # Compute AUC only if voting_type is 'soft'
    try:
        if voting_type == 'soft':
            y_probs = ensemble.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_probs, multi_class=auc_mode, average='macro')
        else:
            auc = None
    except Exception as e:
        print(f"AUC error: {e}")
        auc = None

    results = pd.DataFrame([{
        'Model': f'VotingClassifier ({voting_type})',
        'Accuracy': acc,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1

    }]).set_index('Model')

    return results

## Classification using all faetures
- calling the function

In [27]:
df1.head(3)

,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,RAT_p95,SAO2_min,SAO2_max,SAO2_mean,SAO2_std,SAO2_p5,SAO2_p25,SAO2_p50,SAO2_p75,SAO2_p95
305,S002,307,0.0,-0.000043,0.000020,-2.770550e-07,0.000007,-0.000010,-0.000003,4.394598e-07,...,0.000007,0.969907,0.969941,0.969924,0.000006,0.969915,0.969919,0.969924,0.969930,0.969934
306,S002,308,0.0,-0.000018,0.000014,6.477963e-09,0.000005,-0.000007,-0.000003,4.882887e-08,...,0.000005,0.958276,0.971334,0.966985,0.004638,0.959652,0.959699,0.969916,0.969931,0.969956
307,S002,309,0.0,-0.000050,0.000046,2.981816e-08,0.000008,-0.000011,-0.000003,2.441444e-07,...,0.000006,0.969916,0.969933,0.969924,0.000004,0.969918,0.969920,0.969924,0.969929,0.969931


In [28]:
# Default AUC = 'ovr'
cls_reg_ovr_fusion = multiple_model(df1)
cls_reg_ovr_fusion

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.894234,0.930169,0.873158,0.853763,0.862885
DT,0.827311,0.829073,0.770396,0.769144,0.769659
XGBoost,0.926805,0.985934,0.919471,0.894720,0.906513
LightGBM,0.939349,0.989170,0.932663,0.916377,0.924277
RF,0.911981,0.980086,0.909380,0.858409,0.881184
SVM,0.899437,0.974465,0.885299,0.868085,0.873365


In [29]:
# Or try one-vs-one AUC if OvR seems low
cls_reg_ovo_fusion = multiple_model(df1, auc_mode='ovo')
cls_reg_ovo_fusion

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.894234,0.930431,0.873158,0.853763,0.862885
DT,0.825315,0.825971,0.769773,0.767961,0.768771
XGBoost,0.926805,0.985482,0.919471,0.894720,0.906513
LightGBM,0.939349,0.989374,0.932663,0.916377,0.924277
RF,0.911767,0.979030,0.908508,0.858823,0.881035
SVM,0.899437,0.974591,0.885299,0.868085,0.873365


### Calling Ensemble

In [31]:
# res_ens=ensemble_model(df1)
# res_ens

In [32]:
# # Hard voting — no AUC
# results_hard = ensemble_model(df1, voting_type='hard')
# results_hard

In [33]:
# Soft voting — AUC with OvR
results_soft = ensemble_model(df1, voting_type='soft', auc_mode='ovr')
results_soft

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
VotingClassifier (soft),0.937496,0.987017,0.933258,0.909875,0.921072


In [34]:
# Or try OvO instead
results_soft_ovo = ensemble_model(df1, voting_type='soft', auc_mode='ovo')
results_soft_ovo

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
VotingClassifier (soft),0.937567,0.986902,0.933968,0.909623,0.92126


In [35]:
### Combined_with Ensemble

### OVR

In [37]:
com_ovr=pd.concat([cls_reg_ovr_fusion,results_soft])
com_ovr

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.894234,0.930169,0.873158,0.853763,0.862885
DT,0.827311,0.829073,0.770396,0.769144,0.769659
XGBoost,0.926805,0.985934,0.919471,0.894720,0.906513
LightGBM,0.939349,0.989170,0.932663,0.916377,0.924277
RF,0.911981,0.980086,0.909380,0.858409,0.881184
SVM,0.899437,0.974465,0.885299,0.868085,0.873365
VotingClassifier (soft),0.937496,0.987017,0.933258,0.909875,0.921072


### OvO

In [39]:
com_ovo=pd.concat([cls_reg_ovo_fusion,results_soft_ovo])
com_ovo

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.894234,0.930431,0.873158,0.853763,0.862885
DT,0.825315,0.825971,0.769773,0.767961,0.768771
XGBoost,0.926805,0.985482,0.919471,0.894720,0.906513
LightGBM,0.939349,0.989374,0.932663,0.916377,0.924277
RF,0.911767,0.979030,0.908508,0.858823,0.881035
SVM,0.899437,0.974591,0.885299,0.868085,0.873365
VotingClassifier (soft),0.937567,0.986902,0.933968,0.909623,0.921260
